<a href="https://colab.research.google.com/github/hyymyth69-ops/internship.projects/blob/main/movie_recommendation_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""Content-based movie recommender using TF-IDF and cosine similarity."""

import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def create_sample_movies() -> pd.DataFrame:
    """Return a small built-in dataset so the script works even without a CSV file."""
    return pd.DataFrame(
        [
            {"Title": "Inception", "Genre": "Sci-Fi", "Director": "Christopher Nolan"},
            {"Title": "Interstellar", "Genre": "Sci-Fi", "Director": "Christopher Nolan"},
            {"Title": "The Dark Knight", "Genre": "Action", "Director": "Christopher Nolan"},
            {"Title": "The Matrix", "Genre": "Sci-Fi", "Director": "Lana Wachowski"},
            {"Title": "Fight Club", "Genre": "Drama", "Director": "David Fincher"},
        ]
    )


def load_movies(path: str = "movies.csv") -> pd.DataFrame:
    """Load and clean the movies dataset, or create a fallback dataset if missing."""
    if not os.path.exists(path):
        print(f"{path} not found. Using built-in sample data instead.")
        df = create_sample_movies()
        df.to_csv(path, index=False)
    else:
        df = pd.read_csv(path)

    df.columns = df.columns.str.strip()

    required = {"Title", "Genre", "Director"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"CSV is missing required columns: {missing}")

    df["Genre"] = df["Genre"].fillna("").astype(str).str.strip()
    df["Director"] = df["Director"].fillna("").astype(str).str.strip()
    df["Title"] = df["Title"].astype(str).str.strip()

    df = df.drop_duplicates(subset="Title", keep="first").reset_index(drop=True)
    df["content"] = (df["Genre"] + " " + df["Director"]).str.strip()
    return df


def build_similarity_matrix(df: pd.DataFrame):
    """Vectorize the content column and compute pairwise cosine similarity."""
    tfidf = TfidfVectorizer(stop_words="english")
    tfidf_matrix = tfidf.fit_transform(df["content"])
    return cosine_similarity(tfidf_matrix)


def recommend(movie_name: str, df: pd.DataFrame, similarity, n: int = 5):
    """Return up to n similar movies as a list of (title, score) tuples."""
    lookup = {title.lower(): index for index, title in enumerate(df["Title"])}
    key = movie_name.strip().lower()

    if key not in lookup:
        return None

    idx = lookup[key]
    scores = sorted(enumerate(similarity[idx]), key=lambda item: item[1], reverse=True)
    scores = [score for score in scores if score[0] != idx]
    return [(df.iloc[i]["Title"], round(score, 3)) for i, score in scores[:n]]


def run_tests(df: pd.DataFrame, similarity):
    """Basic sanity checks to confirm the pipeline works correctly."""
    for i in range(len(df)):
        assert abs(similarity[i][i] - 1.0) < 1e-6, f"Self-similarity failed at row {i}"

    assert (similarity - similarity.T < 1e-9).all(), "Similarity matrix is not symmetric"

    result = recommend("inception", df, similarity)
    assert result is not None and len(result) > 0, "Expected recommendations for Inception"

    assert recommend("this movie does not exist", df, similarity) is None

    print("All tests passed ✅")


if __name__ == "__main__":
    movies = load_movies("movies.csv")
    similarity_matrix = build_similarity_matrix(movies)

    run_tests(movies, similarity_matrix)

    movie_name = input("\nEnter movie name: ").strip()
    if not movie_name:
        print("Please enter a movie name.")
    else:
        results = recommend(movie_name, movies, similarity_matrix)
        if results is None:
            print("Movie not found. Check the spelling and try again.")
        else:
            print(f"\nRecommended movies for {movie_name}:\n")
            for title, score in results:
                print(f"{title}  (score: {score})")

movies.csv not found. Using built-in sample data instead.
All tests passed ✅

Enter movie name: inception

Recommended movies for inception:

Interstellar  (score: 1.0)
The Dark Knight  (score: 0.486)
The Matrix  (score: 0.393)
Fight Club  (score: 0.0)
